# IMA205 CHALLENGE MAXIME APPERT

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import csv
import torch
import torch.nn as nn
import torchvision
from torch import optim
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from PIL import Image, ImageDraw, ImageFont
from torchvision.transforms.functional import to_pil_image, to_tensor
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
import cv2
import warnings
warnings.filterwarnings('ignore')

print(pd.__version__)
print(np.__version__)
print(csv.__version__)
print(torch.__version__)
print(torchvision.__version__)
print(Image.__version__)
print(sklearn.__version__)
print(cv2.__version__)


2.1.4
1.26.3
1.0
2.2.2+cu121
0.17.2+cpu
10.2.0
1.4.2
4.9.0


## Weight Analysis

In [84]:
df = pd.read_csv('metadataTrain.csv')

class_counts = df['CLASS'].value_counts()
print(class_counts)

# Calculate weights
total = class_counts.sum()
class_weights = class_counts / total

# Print the weights
print(class_weights)
print(1/class_weights)

CLASS
2    9657
1    3391
3    2492
5    1968
4     650
8     471
7     190
6     179
Name: count, dtype: int64
CLASS
2    0.508317
1    0.178492
3    0.131172
5    0.103590
4    0.034214
8    0.024792
7    0.010001
6    0.009422
Name: count, dtype: float64
CLASS
2      1.967278
1      5.602477
3      7.623596
5      9.653455
4     29.227692
8     40.335456
7     99.989474
6    106.134078
Name: count, dtype: float64


## Segmentation of Images

In [ ]:
def segmentation(image_filename, root='./Train', square_size=75):
    img = cv2.imread(os.path.join(root, image_filename + ".jpg"))
    if os.path.exists(os.path.join(root, image_filename + "_seg.png")):
        seg = cv2.imread(os.path.join(root, image_filename + "_seg.png"), cv2.IMREAD_GRAYSCALE)
    else:
        img_temp = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        im = img_temp > 40
        (r, c) = np.where(im == 1)
        min_row, min_col = np.min(r), np.min(c)
        max_row, max_col = np.max(r), np.max(c)
        im_crop = img[min_row:max_row, min_col:max_col]
        while(im_crop[0,0][0]<10 and im_crop[0,0][1]<10 and im_crop[0,0][2]<10):
            min_row += 1
            min_col += 1
            max_row -= 1
            max_col -= 1
            im_crop = img[min_row:max_row, min_col:max_col]

        img = im_crop
        img_bin = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        _, threshold = cv2.threshold(img_bin,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

        # Use opening to remove hair
        ker = np.ones((2,2),np.uint8)
        opening = cv2.morphologyEx(threshold,cv2.MORPH_OPEN, ker, iterations = 2)

        ker = np.ones((6,6),np.uint8)
        dilate = cv2.dilate(opening,ker,iterations=3)
        blur = cv2.blur(dilate,(15,15))

        #Search for biggest contour
        _, threshold = cv2.threshold(blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(threshold,cv2.RETR_TREE,cv2.CHAIN_APPROX_NONE)
        cnt = max(contours, key=cv2.contourArea)

        # Create segmentation
        seg = np.zeros((img.shape[:2]), np.uint8)
        cv2.drawContours(seg, [cnt],-1, 255, -1)

        # Define the size of the square
        square_size = square_size  # Change this to the desired size

        # Calculate the coordinates of the square
        start_row = seg.shape[0] // 2 - square_size // 2
        start_col = seg.shape[1] // 2 - square_size // 2
        end_row = start_row + square_size
        end_col = start_col + square_size

        # Extract the square from the mask
        square = seg[start_row:end_row, start_col:end_col]

        # Check if all pixels in the square are black
        if np.all(square == 0):
            # If all pixels are black, invert the square
            seg = 255 - seg


    res = cv2.bitwise_and(img, img, mask=seg)

    # Save the result
    res_rgb = cv2.cvtColor(res, cv2.COLOR_BGR2RGB)
    res_pil = Image.fromarray(res_rgb)
    res_pil.save(os.path.join(root+'_Seg', image_filename + ".jpg"))
    return

In [ ]:
df = pd.read_csv('metadataTrain.csv')
for id in df['ID']:
    segmentation(id)

df = pd.read_csv('metadataTest.csv')
for id in df['ID']:
    segmentation(id, root='./Test')

## Loading Data

In [2]:
class SkinLesionDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform, train=True):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.train = train

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_name = os.path.join(self.root_dir, str(self.data_frame.iloc[idx, 0]) + ".jpg")
        image = Image.open(img_name)
        image = self.transform(image)

        if self.train:
            label = self.data_frame.iloc[idx, 1] - 1
            return image, label
        else:
            return image, str(self.data_frame.iloc[idx, 0])
        

In [3]:
train_csv = "./metadataTrain.csv"
test_csv = "./metadataTest.csv"
train_root = "./Train"
test_root = "./Test"
#train_root = "./Train_Seg"             #To use segmented images instead of original images
#test_root = "./Test_Seg"

# Transformations (resize and some data augmentation)
transform_train = torchvision.transforms.Compose([
    torchvision.transforms.GaussianBlur(3, sigma=(0.1, 2.0)),   #Data augmentation
    torchvision.transforms.RandomRotation(90),  #Data augmentation
    torchvision.transforms.RandomHorizontalFlip(),  #Data augmentation
    torchvision.transforms.Resize((224,224)),       #ResNet generally requires 224x224 images
    torchvision.transforms.ToTensor(),       #Convert to tensor
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]) #Mean and std from ImageNet based on https://pytorch.org/vision/stable/models.html

transform_test = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224,224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

train_dataset = SkinLesionDataset(csv_file=train_csv, root_dir=train_root, transform=transform_train)
test_dataset = SkinLesionDataset(csv_file=test_csv, root_dir=test_root, transform=transform_test, train=False)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)


## Convolutional Neural Network

In [4]:
model = torchvision.models.resnet18(pretrained=True)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
num_classes = 8 
model.fc = torch.nn.Linear(512, num_classes)
model.to(device)
learning_rate = 0.00001 
criterion = nn.CrossEntropyLoss(weight=torch.tensor([5.602477, 1.967278, 7.623596, 29.227692, 9.653455, 106.134078, 99.989474, 40.335456]).to(device))
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
     

cuda:0


In [5]:
def vector_to_class(x):
  y = torch.argmax(x,axis=1)
  return y

def prediction_accuracy(predict,labels):
  accuracy = (predict == labels).sum()/(labels.shape[0])
  return accuracy

In [6]:
#Code directly from MLP TP part 3
n_epochs = 12
for epoch in range(0,n_epochs):
  train_loss=0.0
  all_labels = []
  all_predicted = []

  with tqdm(train_loader, unit="batch") as tepoch:
    for data, labels in tepoch:
      tepoch.set_description(f"Epoch {epoch}")
      # Put the data on device
      data = data.to(device)
      labels = labels.to(device)

      y_predict = model(data)
      loss = criterion(y_predict, labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # Compute the loss
      train_loss += loss.item()
      # Store labels and class predictions
      all_labels.extend(labels.tolist())
      all_predicted.extend(vector_to_class(y_predict).tolist())

  print('Epoch {}: Train Loss: {:.4f}'.format(epoch, train_loss/len(train_loader.dataset)))
  print('Epoch {}: Train Accuracy: {:.4f}'.format(epoch, prediction_accuracy(np.array(all_predicted),np.array(all_labels))))

Epoch 0: 100%|██████████| 594/594 [04:07<00:00,  2.40batch/s]


Epoch 0: Train Loss: 0.0535
Epoch 0: Train Accuracy: 0.4060


Epoch 1: 100%|██████████| 594/594 [03:40<00:00,  2.69batch/s]


Epoch 1: Train Loss: 0.0390
Epoch 1: Train Accuracy: 0.5914


Epoch 2: 100%|██████████| 594/594 [03:41<00:00,  2.69batch/s]


...

Epoch 9: Train Loss: 0.0223
Epoch 9: Train Accuracy: 0.7096


Epoch 10: 100%|██████████| 594/594 [03:38<00:00,  2.72batch/s]


Epoch 10: Train Loss: 0.0215
Epoch 10: Train Accuracy: 0.7172


Epoch 11: 100%|██████████| 594/594 [03:59<00:00,  2.48batch/s]

Epoch 11: Train Loss: 0.0203
Epoch 11: Train Accuracy: 0.7242


## Metadata Training

In [124]:
#Naive first attempt at implementation of the metadata classifier, not to run! Only to show what I frist tried to do.

# train_data = pd.read_csv('metadataTrain.csv')
# test_data = pd.read_csv('metadataTest.csv')
# # Calculate the distributions of SEX, AGE, and POSITION in the training data
# sex_distribution = train_data['SEX'].value_counts(normalize=True)
# age_distribution = train_data['AGE'].value_counts(normalize=True)
# position_distribution = train_data['POSITION'].value_counts(normalize=True)
# sex_mapping = {'male': 1, 'female': 0}
# train_data['SEX'] = train_data['SEX'].map(sex_mapping).fillna(0).astype(int)
# test_data['SEX'] = test_data['SEX'].map(sex_mapping).fillna(0).astype(int)
# # Encode the SEX and POSITION columns as integers
# le = LabelEncoder()
# train_data['POSITION'] = le.fit_transform(train_data['POSITION'])
# test_data['POSITION'] = le.transform(test_data['POSITION'])

# # Calculate the conditional probabilities of each class given each SEX, AGE, and POSITION
# sex_conditional_probabilities = train_data.groupby('SEX')['CLASS'].value_counts(normalize=True)
# age_conditional_probabilities = train_data.groupby('AGE')['CLASS'].value_counts(normalize=True)
# position_conditional_probabilities = train_data.groupby('POSITION')['CLASS'].value_counts(normalize=True)

In [7]:
train_data = pd.read_csv('metadataTrain.csv')
test_data = pd.read_csv('metadataTest.csv')
sex_mapping = {'male': 1, 'female': 0}
train_data['SEX'] = train_data['SEX'].map(sex_mapping).fillna(0).astype(int)
test_data['SEX'] = test_data['SEX'].map(sex_mapping).fillna(0).astype(int)
# Encode the SEX and POSITION columns as integers
le = LabelEncoder()
train_data['POSITION'] = le.fit_transform(train_data['POSITION'])
test_data['POSITION'] = le.transform(test_data['POSITION'])
features = train_data[['SEX', 'AGE', 'POSITION']]
target = train_data['CLASS']

# Train a decision tree
tree = DecisionTreeClassifier()
tree.fit(features, target)

# Calculate probabilities
probabilities = tree.predict_proba(test_data[['SEX', 'AGE', 'POSITION']])
print(probabilities)

[[0.2248996  0.58634538 0.01204819 ... 0.00803213 0.00803213 0.02811245]
 [0.1322314  0.50413223 0.18181818 ... 0.03305785 0.         0.02479339]
 [0.         0.55555556 0.         ... 0.         0.         0.        ]
 ...
 [0.         1.         0.         ... 0.         0.         0.        ]
 [0.28671329 0.25874126 0.23776224 ... 0.         0.         0.        ]
 [0.04242424 0.85454545 0.06666667 ... 0.         0.         0.        ]]


## Test Set Prediction

In [8]:
test_data_df = pd.DataFrame(test_data)
test_data_df.set_index('ID', inplace=True)
distribution = [0.7005531, 0.24592265, 0.95261733, 3.64804147, 1.20674543, 13.19375, 12.56547619, 5.04219745]
distribution = [x / sum(distribution) for x in distribution]  # Normalize to sum to 1

all_predicted = []

with tqdm(test_loader) as tepoch:
    for data, id in tepoch:
        data = data.to(device)
        y_predict = model(data)
        y_predict = y_predict.softmax(dim=1)
        y_predict = y_predict * torch.tensor(distribution).to(device)  # Adjust based on test set distribution
        for i in range(len(id)):
            metadata = test_data_df.loc[id[i]]
            tree_probs = tree.predict_proba([metadata.values]).squeeze()
            # Multiply the model's predictions by the tree's probabilities
            y_predict[i] *= torch.tensor(tree_probs).to(device)
        y_predict = y_predict.argmax(dim=1)
        all_predicted.extend(zip(y_predict.tolist(), id))    

# Save results to CSV file
with open('test_results.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['ID', 'CLASS'])
    for pred_class, id in all_predicted:
        writer.writerow([id, pred_class+1])

100%|██████████| 198/198 [00:28<00:00,  6.87it/s]
